In [1]:
import sys
import warnings

sys.path.append("..")
sys.dont_write_bytecode = True
warnings.filterwarnings("ignore")

In [2]:
import polars as pl
from utils.utils import (
    split_train_test,
    get_parent_run_id_from_experiment,
    compute_generalisation_error_from_run_id_and_experiment_id,
)
from utils.statics import lightgbm_model_name, FEATUERE_ENGINEERING_EXPERIMENT_ID
from utils.preprocessing_handler import PreprocessingHandler
from feature_engineering.RowWiseTransformations import RowWiseTransformations
from feature_engineering.OpenFETransformations import OpenFETransformations
from feature_engineering.ColumnTransformer import ColumnTransformer
from model.nested_cv_eval import ModelCVEvaluator

model_type = lightgbm_model_name

In [3]:
passes_df = pl.read_parquet("../data/02-analysis/passes.parquet")

In [4]:
train_df, test_df = split_train_test(passes_df=passes_df)

In [5]:
numerical_columns = [
    "start_x",
    "start_y",
    "end_x",
    "end_y",
    "length",
    "angle",
    "duration",
]

columns = train_df.columns
target_column = "outcome"
categorical_columns = [col for col in columns if col not in numerical_columns and col != target_column]

preprocessing_handler = PreprocessingHandler(df=train_df, categorical_columns=categorical_columns)
train_df_temp = preprocessing_handler.preprocess_categorical_columns()
train_df_temp = preprocessing_handler.preprocess_outcome_column()
y_train = train_df_temp.select("outcome").to_series()
X_train = preprocessing_handler.preprocess_outcome_column().drop(pl.col("outcome"))

In [6]:
categorical_columns = ["body_part", "height"]
column_transformer_config = {
            "cat_columns": categorical_columns
        }

In [7]:
row_wise_transformations = RowWiseTransformations()
column_wise_transformations = ColumnTransformer(**column_transformer_config)
open_fe_transformations = OpenFETransformations(n_features=5)

In [8]:
run_name = f"Feature_engineering_11_OFE"
experiment_name = "Feature Engineering"
model_cv_evaluator = ModelCVEvaluator(
    model_type=model_type,
    row_wise_transformations=row_wise_transformations,
    column_wise_transformations=column_wise_transformations,
    open_fe_transformations=open_fe_transformations,
    n_inner_splits=3,
    n_outer_splits=2,
    n_trials=50,
    use_hyperparameter_tuning=False,
    use_feature_selection=False,
    use_feature_engineering=True,
    use_ofe=True,
    log_feature_importance=True,
    log_parameter_importance=True,
    run_name=run_name,
    experiment_name=experiment_name,
    categorical_columns=categorical_columns,
)

In [9]:
result_cv = model_cv_evaluator.get_generalisation_error(X_train=X_train, y_train=y_train)

2026-04-26 14:50:34,645 - INFO - Starting training with model lightgbm with the following configuration:
        - 3 inner splits
        - 2 outer splits
        - 50 trials
2026-04-26 14:50:35,565 - INFO - Fitting OpenFE on fold 1


The number of candidate features is 1775
Start stage I selection.


100%|██████████| 32/32 [00:18<00:00,  1.73it/s]


463 same features have been deleted.
Meet early-stopping in successive feature-wise halving.


100%|██████████| 32/32 [00:52<00:00,  1.64s/it]


...

100%|██████████| 32/32 [00:23<00:00,  1.37it/s]


Finish data processing.


2026-04-26 14:54:36,025 - INFO - Feature: autoFE_f_0           | Formula: (duration+under_pressure)
2026-04-26 14:54:36,025 - INFO - Feature: autoFE_f_1           | Formula: (duration-angle_cos)
2026-04-26 14:54:36,025 - INFO - Feature: autoFE_f_2           | Formula: (length/direction_to_goal_cos)
2026-04-26 14:54:36,025 - INFO - Feature: ofe_col_1            | Formula: freq(duration)
2026-04-26 14:54:36,034 - INFO - Feature: ofe_col_2            | Formula: GroupByThenRank(duration,start_y)
2026-04-26 14:54:36,037 - INFO - Starting full hyperparameter tuning...
2026-04-26 14:54:36,037 - INFO - Fitting final model with best hyperparameters...


['body_part', 'height']


🏃 View run Outer_fold_2 at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/3422a2685ece4602ae346999554a5b15
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708
🏃 View run Feature_engineering_11_OFE_lightgbm at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/ee29584d638c465ea84c3d6875fd590a
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708


In [10]:
parent_run_id = get_parent_run_id_from_experiment(result=result_cv, experiment_id=FEATUERE_ENGINEERING_EXPERIMENT_ID)
compute_generalisation_error_from_run_id_and_experiment_id(
    parent_run_id=parent_run_id, experiment_id=FEATUERE_ENGINEERING_EXPERIMENT_ID
)

Number of outer folds: 3
95% confidence interval for best estimate of generalisation: 0.2534289568910852 ± 0.00277424530519379


# Using node features

In [10]:
X_train_pd = X_train.to_pandas().copy()
y_train_pd = y_train.to_pandas().copy()

In [11]:
from sklearn.model_selection import train_test_split
import pandas as pd

_, _, train_y, test_y = train_test_split(
                    X_train_pd,
                    y_train_pd,
                    stratify=y_train_pd,
                    test_size=0.2,
                    random_state=1)

train_idx = train_y.index
test_idx = test_y.index

In [12]:
row_wise, column_wise, ofe = open_fe_transformations.fit(X_train_pd.loc[train_idx], y_train_pd.loc[train_idx], task='classification', categorical_features=categorical_columns, feature_boosting=True, n_jobs=1)

The number of candidate features is 499
Start stage I selection.


100%|██████████| 4/4 [00:22<00:00,  5.64s/it]


166 same features have been deleted.
Meet early-stopping in successive feature-wise halving.


100%|██████████| 4/4 [01:25<00:00, 21.30s/it]


The number of remaining candidate features is 157
Start stage II selection.


100%|██████████| 4/4 [00:38<00:00,  9.54s/it]


Finish data processing.


In [13]:
df_val = pd.DataFrame([])

In [14]:
X_train_transformed, _, mapping = open_fe_transformations.apply_openfe_features(column_wise, X_train_pd.loc[train_idx], df_val, n_jobs=1)

In [15]:
X_train_transformed

,start_x,start_y,end_x,end_y,length,height,angle,duration,body_part,under_pressure,autoFE_f_0,autoFE_f_1,autoFE_f_2
43926,66.0,44.0,80.0,40.0,14.560220,High Pass,-0.278300,0.920,Right Foot,0,774.0,0.063815,0.191601
10172,28.0,28.0,25.0,22.0,6.708204,Low Pass,-2.034444,1.640,Head,0,482.0,0.713699,0.702593
49317,31.0,8.0,46.0,5.0,15.297058,Ground Pass,-0.197396,1.573,Left Foot,0,241.0,0.749037,0.639344
9550,24.0,34.0,57.0,4.0,44.598206,High Pass,-0.737815,2.533,Right Foot,0,62.0,0.638518,0.894399
30594,25.0,51.0,27.0,19.0,32.062440,Ground Pass,-1.508378,1.840,Right Foot,0,304.0,0.855345,0.781250
...,...,...,...,...,...,...,...,...,...,...,...,...,...
50870,16.0,51.0,17.0,43.0,8.062258,Ground Pass,-1.446441,0.840,Right Foot,0,769.0,0.215846,0.147368
25978,22.0,59.0,66.0,78.0,47.927030,High Pass,0.407632,2.307,Right Foot,1,17.0,0.571912,0.851286
48288,100.0,5.0,98.0,13.0,8.246211,High Pass,1.815775,0.653,Unknown,1,214.0,0.033532,0.111399
13506,82.0,71.0,92.0,78.0,12.206555,Ground Pass,0.610726,0.973,Right Foot,0,448.0,0.329612,0.283360


In [16]:
mapping

{'autoFE_f_0': 'freq(duration)',
 'autoFE_f_1': 'GroupByThenRank(duration,height)',
 'autoFE_f_2': 'GroupByThenRank(duration,end_y)'}

In [17]:
from feature_engineering.OpenFE.FeatureGenerator import Node, FNode

mean_node = Node("freq", [FNode("duration")])
nodes = [mean_node]

In [18]:
column_wise_transformations.fit(X_train_pd.loc[train_idx], feature_nodes=nodes)
X_train_transformed = column_wise_transformations.fit_transform(X_train_pd.loc[train_idx], feature_nodes=nodes)

In [19]:
X_train_transformed

,start_x,start_y,end_x,end_y,length,height,angle,duration,body_part,under_pressure,freq(duration)
43926,66.0,44.0,80.0,40.0,14.560220,High Pass,-0.278300,0.920,Right Foot,0,774.0
10172,28.0,28.0,25.0,22.0,6.708204,Low Pass,-2.034444,1.640,Head,0,482.0
49317,31.0,8.0,46.0,5.0,15.297058,Ground Pass,-0.197396,1.573,Left Foot,0,241.0
9550,24.0,34.0,57.0,4.0,44.598206,High Pass,-0.737815,2.533,Right Foot,0,62.0
30594,25.0,51.0,27.0,19.0,32.062440,Ground Pass,-1.508378,1.840,Right Foot,0,304.0
...,...,...,...,...,...,...,...,...,...,...,...
50870,16.0,51.0,17.0,43.0,8.062258,Ground Pass,-1.446441,0.840,Right Foot,0,769.0
25978,22.0,59.0,66.0,78.0,47.927030,High Pass,0.407632,2.307,Right Foot,1,17.0
48288,100.0,5.0,98.0,13.0,8.246211,High Pass,1.815775,0.653,Unknown,1,214.0
13506,82.0,71.0,92.0,78.0,12.206555,Ground Pass,0.610726,0.973,Right Foot,0,448.0
